# Tools for Forecasting Rat Sightings

In this notebook, there are two models which one can use to forecast rat sightings citywide in NYC. They are:

(a) is the Prophet model with confidence inverals included to provide the user with some idea of the range of possible rat sightings on any given day.

(b) is the NeuralProphet model which offers slight improvements to the Prophet model by incorporating weather data.

# Prophet: 14 Days

## Import Packages

In [137]:
import requests 
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging



## Set Parameters

In [138]:
params = {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5,}

## Download the Rat Sightings Data

In [139]:
## Turn this on if you want to download the most recent data to use. 
## Be careful! The database is updated on a daily basis and so may not be complete.

# # We downloads the rat sightings data to a folder.

# import requests

# url = "https://data.cityofnewyork.us/api/views/3q43-55fe/rows.csv?accessType=DOWNLOAD"

# response = requests.get(url)
# response.raise_for_status()

# with open("../../scr/data/rat_sightings_data/Rat_Sightings_NYC.csv", "wb") as f:
#     f.write(response.content)

## Prepare the Rat Sightings Data

In [140]:
# Open rat sightings data
rat_sighting = pd.read_csv("../../scr/data/rat_sightings_data/Rat_Sightings_NYC.csv")



# Clean the data
rat_sighting.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rat_sighting.columns] #apply to column headers
rat_sighting['location_type'] = rat_sighting['location_type'].str.strip().str.replace(' ', '_').str.lower()  #apply to location_type column
cols_to_drop = [c for c in rat_sighting.columns if (rat_sighting[c].nunique(dropna=False) == 1)]
rat_sighting = rat_sighting.drop(columns=cols_to_drop)
rat_sighting['created_date'] = pd.to_datetime(rat_sighting['created_date']) 
rat_sighting = rat_sighting.drop(columns='park_borough')
rat_sighting = rat_sighting.drop(columns=['location'])
rs = rat_sighting.copy()
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


WARNING - (py.warnings._showwarnmsg) - /var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_48662/1036270658.py:11: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.




In [141]:
last_date = rs['ds'].max()
end_date   = last_date.strftime("%Y-%m-%d")

date_range = pd.date_range(start="2020-01-01", end=end_date)

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({'holiday': 'federal_us', 'ds': pd.to_datetime(holidays), 'lower_window': 0, 'upper_window': 1})

holidays = federal_holidays

## Train the Model

We set interval_width = 0.5 so that in forecasting, yhat_lower and yhat_upper will indicate the confidence level of 50%

Why did we pick such a low confidence interval? During experimentaiton, we found that Prophet produced very large confidence intervals. One can already see this by adjusting interval_width=0.5 to interval_width=0.8 which is the default. From looking at the plots, it appears that Prophet is trying to account random spikes and dips in rat sightings reports during the summer and winters. For 14 day forecasting purposes, a confidence level of 50% seems to provide more refined confidence intervals for use. We also observe that Prophet is overcompensating with higher confidence levels and prediction *negative* values for yhat_lower. We avoid this by choosing confidence level 50%. We cal also avoid this by taking the minimum of yhat_lower and 0.

In [142]:
m = Prophet(**params, interval_width=0.5)
m.add_country_holidays(country_name='US')
m.fit(rs)

## Forecast for 14 Days

In [143]:
future = m.make_future_dataframe(periods=14)
forecast = m.predict(future)
forecast

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,Christmas Day,Christmas Day_lower,Christmas Day_upper,Christmas Day (observed),...,weekly,weekly_lower,weekly_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2020-01-01,43.140229,8.096491,26.015909,43.140229,43.140229,0.0,0.0,0.0,0.0,...,7.376204,7.376204,7.376204,-20.816978,-20.816978,-20.816978,0.0,0.0,0.0,16.961252
1,2020-01-02,43.129834,21.155004,38.966617,43.129834,43.129834,0.0,0.0,0.0,0.0,...,6.544724,6.544724,6.544724,-20.124212,-20.124212,-20.124212,0.0,0.0,0.0,29.550346
2,2020-01-03,43.119440,13.466005,32.468171,43.119440,43.119440,0.0,0.0,0.0,0.0,...,-0.852028,-0.852028,-0.852028,-19.413993,-19.413993,-19.413993,0.0,0.0,0.0,22.853418
3,2020-01-04,43.109045,-3.114706,15.720459,43.109045,43.109045,0.0,0.0,0.0,0.0,...,-17.843521,-17.843521,-17.843521,-18.695953,-18.695953,-18.695953,0.0,0.0,0.0,6.569571
4,2020-01-05,43.098650,-0.182860,18.623822,43.098650,43.098650,0.0,0.0,0.0,0.0,...,-16.014918,-16.014918,-16.014918,-17.980035,-17.980035,-17.980035,0.0,0.0,0.0,9.103697
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2268,2026-03-18,42.950950,34.088229,52.184612,42.950950,42.950950,0.0,0.0,0.0,0.0,...,7.376204,7.376204,7.376204,-7.260861,-7.260861,-7.260861,0.0,0.0,0.0,43.066293
2269,2026-03-19,42.903713,33.041062,51.128513,42.903713,42.903713,0.0,0.0,0.0,0.0,...,6.544724,6.544724,6.544724,-7.377100,-7.377100,-7.377100,0.0,0.0,0.0,42.071337
2270,2026-03-20,42.856476,26.509402,43.077000,42.856476,42.856476,0.0,0.0,0.0,0.0,...,-0.852028,-0.852028,-0.852028,-7.481272,-7.481272,-7.481272,0.0,0.0,0.0,34.523176
2271,2026-03-21,42.809239,8.855613,26.809126,42.809239,42.809239,0.0,0.0,0.0,0.0,...,-17.843521,-17.843521,-17.843521,-7.569863,-7.569863,-7.569863,0.0,0.0,0.0,17.395856


In [144]:
predictions = forecast.tail(14).copy()
cols = ['yhat', 'yhat_lower', 'yhat_upper']
predictions.loc[:, cols] = (predictions[cols].round().clip(lower=0).astype(int))
predictions = predictions[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

predictions

,ds,yhat,yhat_lower,yhat_upper
2259,2026-03-09,49.0,39.0,58.0
2260,2026-03-10,46.0,37.0,55.0
2261,2026-03-11,44.0,35.0,53.0
2262,2026-03-12,43.0,33.0,52.0
2263,2026-03-13,36.0,27.0,46.0
2264,2026-03-14,19.0,10.0,28.0
2265,2026-03-15,20.0,10.0,29.0
2266,2026-03-16,48.0,39.0,57.0
2267,2026-03-17,45.0,35.0,54.0
2268,2026-03-18,43.0,34.0,52.0


## Plots

In [145]:
fig1 = plot_plotly(m, forecast)
fig1.show()

fig2 = plot_components_plotly(m, forecast)
fig2.show()

WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result




WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result


WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result


WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprec

## Hyperparameter Tuning

Since there are only so many hyperparameters to tune, we use a grid search. In the following code block, we have param_grid which has already saved the best values we found on our tests changepoint_prior_scale=0.1 and seasonality_prior_scale=5. The user can add more to these arrays to try and find better parameters if they would like.

In [146]:
# param_grid = {  
#     'changepoint_prior_scale': [0.1],
#     'seasonality_prior_scale': [5],
# }

# # Open rat sightings data
# rat_sighting = pd.read_csv("../../scr/data/rat_sightings_data/Rat_Sightings_NYC.csv")



# # Clean the data
# rat_sighting.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rat_sighting.columns] #apply to column headers
# rat_sighting['location_type'] = rat_sighting['location_type'].str.strip().str.replace(' ', '_').str.lower()  #apply to location_type column
# cols_to_drop = [c for c in rat_sighting.columns if (rat_sighting[c].nunique(dropna=False) == 1)]
# rat_sighting = rat_sighting.drop(columns=cols_to_drop)
# rat_sighting['created_date'] = pd.to_datetime(rat_sighting['created_date']) 
# rat_sighting = rat_sighting.drop(columns='park_borough')
# rat_sighting = rat_sighting.drop(columns=['location'])
# rs = rat_sighting.copy()
# rs['created_date'] = pd.to_datetime(rs['created_date']) 
# rs = rs[rs['created_date']>='2020-01-01']
# rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
# rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

# df = rs.copy()

# ## To tune for hyperparameters, add more possible parameters to the dictionary below and add more values to it.
# ## So far, the I've been able to get is {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5}

# init_days = '2054 days'
# cv_period = '14 days'
# forecast_horizon = '14 days'


# # Generate all combinations of parameters
# all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
# rmses = []  # Store the RMSEs for each params here
# performance = []

# # Use cross validation to evaluate all parameters
# for params in all_params:
#     params['holidays'] = holidays
#     m = Prophet(**params).fit(df)  # Fit model with given params
#     df_cv = cross_validation(m, initial = init_days, period=cv_period, horizon = forecast_horizon)
#     df_p = performance_metrics(df_cv, rolling_window=14)
#     performance.append(df_p)
#     rmses.append(df_p['rmse'].values[0])

# # Find the best parameters
# tuning_results = pd.DataFrame(all_params)
# tuning_results['rmse'] = rmses

# best_params = all_params[np.argmin(rmses)]
# best_params

# NeuralProphet: 14 days

This code block makes 14 day forecasts for the number of rat sightings in NYC.

## Instructions

The model uses on weather data and [data on rat sightings](https://data.cityofnewyork.us/Social-Services/Rat-Sightings/3q43-55fe/about_data) up to the last time the rat sightings dataset was updated. One simply runs all the code blocks in the given section.

0. Set the parameters to be used for the Neural Prophet model. There are some baseline parameters which were obtained from using Optuna. Change these parameters based on need. Tuning hyperparameters takes time and we recommend only doing so if the performance of the model is being affected by suboptimal choices of parameters.

1. Gather the data. There is a codeblock which downloads [NYC Open Data's Rat Sightings Dataset](https://data.cityofnewyork.us/Social-Services/Rat-Sightings/3q43-55fe/about_data) and the weather data up to the last day. One can turn this off by commenting away this section of the code and just use the already saved data. Currently, that codeblock is commented out. For the forecasting, we assume that the weather on the last day will remain the same for the next 14 days.

3. Train a Neural Prophet model on all available data.

4. Forecasts 14 days out. We included plots of the model when fitted to the training data & plots of the model's forecast. The user should use this while tuning the parameters.

## Warnings. 

1. Some fine tuning of the parameters might be needed. We quote from the NeuralProphet documentation: "NeuralProphet is fit with stochastic gradient descent - more precisely, with an AdamW optimizer and a One-Cycle policy. If the parameter learning_rate is not specified, a learning rate range test is conducted to determine the optimal learning rate. The epochs, loss_func and optimizer are other parameters that directly affect the model training process. If not defined, epochsand loss_func are automatically set based on the dataset size. They are set in a manner that controls the total number training steps to be around 1000 to 4000. NeuralProphet offers to set two different values for optimizer, namely AdamW and SDG (stochastic gradient descent). **If it looks like the model is overfitting to the training data (the live loss plot can be useful hereby), you can reduce epochs and learning_rate, and potentially increase the batch_size. If it is underfitting, the number of epochs and learning_rate can be increased and the batch_size potentially decreased.**"

2. The NeuralProphet package appears to no longer be in development. We use the most current version available here.

3. The run time for the modeling process is quite high.

## Set Parameters

In [121]:
parameters = dict()
parameters['apparent_temperature_max'] = 30
parameters['apparent_temperature_min'] = 5
parameters['snowfall_sum'] = 1
parameters['n_lags'] = 16 # 16 is original
parameters['epochs'] = 60 # 60 original
parameters['learning_rate'] = 0.2986532324899507 # 0.2986532324899507  original
parameters['batch_size'] = 128 # 128 original
parameters['ar_reg'] = 2.132925719127823



parameters['n_forecasts'] = 14 

params = parameters.copy()

The defailt settings are

{'lag_temp_max': 30, 
'lag_temp_min': 5, 
'lag_snowfall': 1, 
'n_lags': 16, 
'epochs': 60, 
'learning_rate': 0.2986532324899507, 
'batch_size': 128, 
'ar_reg': 2.132925719127823,
'n_forecasts': 14}

## Import Packages

In [ ]:
import requests 
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging



## Download Rat Sightings Data

In [123]:
## Turn this on if you want to download the most recent data to use. 
## Be careful! The database is updated on a daily basis and so may not be complete.

# # We downloads the rat sightings data to a folder.

# import requests

# url = "https://data.cityofnewyork.us/api/views/3q43-55fe/rows.csv?accessType=DOWNLOAD"

# response = requests.get(url)
# response.raise_for_status()

# with open("../../scr/data/rat_sightings_data/Rat_Sightings_NYC.csv", "wb") as f:
#     f.write(response.content)

## Prepare the Rat Sightings Data

In [124]:
rat_sighting = pd.read_csv("../../scr/data/rat_sightings_data/Rat_Sightings_NYC.csv")


In [125]:
rat_sighting.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rat_sighting.columns] #apply to column headers
rat_sighting['location_type'] = rat_sighting['location_type'].str.strip().str.replace(' ', '_').str.lower()  #apply to location_type column
cols_to_drop = [c for c in rat_sighting.columns if (rat_sighting[c].nunique(dropna=False) == 1)]
rat_sighting = rat_sighting.drop(columns=cols_to_drop)
rat_sighting['created_date'] = pd.to_datetime(rat_sighting['created_date']) 
rat_sighting = rat_sighting.drop(columns='park_borough')
rat_sighting = rat_sighting.drop(columns=['location'])
rs = rat_sighting.copy()
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
rs

WARNING - (py.warnings._showwarnmsg) - /var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_48662/1126600833.py:5: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.




,ds,y
0,2020-01-01,17
1,2020-01-02,40
2,2020-01-03,41
3,2020-01-04,25
4,2020-01-05,17
...,...,...
2254,2026-03-04,45
2255,2026-03-05,55
2256,2026-03-06,47
2257,2026-03-07,39


## Download the Weather Data

The forecasted weather values are the naive forecast i.e. we take the last observed day and assume that these are good predictions for the next 14 days. Better weather data might help improve the model.

In [134]:
lat, lon = 40.7831, -73.9712
last_date = rs['ds'].max()
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)

# # weather data in wd_14 should be the same as the last row of wd. 
# # check the displayed dataframe that this is the case

# display(wd.tail(2))
# display(wd_14)


In [127]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)


regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])

rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")

## Fixing the Seed

We fix the seed for reproducibility. W

Without the following code block, the forecasts and *will* change on each run.

In [ ]:
import random
import torch


# set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# make PyTorch deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Train the model

In [129]:
from neuralprophet import NeuralProphet
import pandas as pd

forecast_horizon = 14
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

model = NeuralProphet(yearly_seasonality=True, 
                      weekly_seasonality=True, 
                      epochs=params['epochs'], 
                      learning_rate=params['learning_rate'], 
                      batch_size=params['batch_size'], 
                      n_forecasts = params['n_forecasts'], 
                      n_lags = params['n_lags'], 
                      ar_reg = params['ar_reg'], 
                      )
model = model.add_country_holidays(country_name="US")

for col in regressed_features:
    model.add_lagged_regressor(col, n_lags=params[col])

model.fit(rs, freq="D")
future = model.make_future_dataframe(rs, periods=forecast_horizon, n_historic_predictions=28)

for col in regressed_features:
    future[col] = pd.concat([rs[col], wd_14[col]], ignore_index=True)

forecast = model.predict(future)

rs_df = model.get_latest_forecast(forecast, include_previous_forecasts = 14)

rs_df

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning:

MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.




Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/data/split.py:273: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 98.611% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 98.611% of the data.
INFO - (NP.df_ut

Predicting: 18it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


,ds,y,origin-14,origin-13,origin-12,origin-11,origin-10,origin-9,origin-8,origin-7,origin-6,origin-5,origin-4,origin-3,origin-2,origin-1,origin-0
0,2026-02-23,10.0,48.299576,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,2026-02-24,26.0,45.122021,32.912827,None,None,None,None,None,None,None,None,None,None,None,None,None
2,2026-02-25,30.0,43.569351,39.680328,34.714645,None,None,None,None,None,None,None,None,None,None,None,None
3,2026-02-26,40.0,41.020496,36.085423,35.31802,38.146759,None,None,None,None,None,None,None,None,None,None,None
4,2026-02-27,38.0,38.897892,36.090687,33.86377,36.580273,34.069916,None,None,None,None,None,None,None,None,None,None
5,2026-02-28,23.0,21.308903,19.646049,17.143513,17.715206,18.434055,19.248264,None,None,None,None,None,None,None,None,None
6,2026-03-01,26.0,22.404171,18.670811,17.887119,19.737465,18.120071,18.962875,19.189486,None,None,None,None,None,None,None,None
7,2026-03-02,40.0,44.845592,42.369926,41.733768,42.604198,41.409336,40.834938,40.642326,42.423771,None,None,None,None,None,None,None
8,2026-03-03,34.0,43.315968,42.395317,40.656109,41.415596,37.994843,38.994373,39.547535,40.955822,38.545338,None,None,None,None,None,None
9,2026-03-04,45.0,42.117523,40.319344,39.23061,39.082008,38.430161,39.828857,41.489872,38.962559,35.928013,36.878204,None,None,None,None,None


### How to read the table.
The columns "origin-i" corresponds to the predicted value yhat for y that was made i steps ago. This happens because we are asking the model to make 14 day forecasts at each step. For the actual forecasting into the future, we take the column "origin-0" for which there are 14 non-None values starting from the first day we wish to forecast. 

## Forecast for 14 Days

In [130]:
df_wide= rs_df.copy()

# use the last 14 entries of 'origin-0' as the forecast
forecast_horizon = 14
origin_col = 'origin-0' 
last_14_preds = df_wide[origin_col].dropna().tail(forecast_horizon)

# compute corresponding forecast dates
last_14_dates = pd.to_datetime(df_wide['ds'].tail(forecast_horizon))

forecast_vertical = pd.DataFrame({'ds': last_14_dates, 'yhat': last_14_preds.values})

for _, row in forecast_vertical.iterrows():
    print(f"On day {row['ds'].date()}, the model predicts {round(row['yhat'])} rat sightings reported to 311.\n")

On day 2026-03-09, the model predicts 38 rat sightings reported to 311.

On day 2026-03-10, the model predicts 40 rat sightings reported to 311.

On day 2026-03-11, the model predicts 39 rat sightings reported to 311.

On day 2026-03-12, the model predicts 41 rat sightings reported to 311.

On day 2026-03-13, the model predicts 37 rat sightings reported to 311.

On day 2026-03-14, the model predicts 20 rat sightings reported to 311.

On day 2026-03-15, the model predicts 22 rat sightings reported to 311.

On day 2026-03-16, the model predicts 45 rat sightings reported to 311.

On day 2026-03-17, the model predicts 41 rat sightings reported to 311.

On day 2026-03-18, the model predicts 42 rat sightings reported to 311.

On day 2026-03-19, the model predicts 47 rat sightings reported to 311.

On day 2026-03-20, the model predicts 36 rat sightings reported to 311.

On day 2026-03-21, the model predicts 24 rat sightings reported to 311.

On day 2026-03-22, the model predicts 22 rat sighti

**Example: On March 9th @ 11AM CST, I ran the above and got the following predictions.**

On day 2026-03-09, the model predicts 38 rat sightings reported to 311.

**Update:** As of checking the data on 2026-03-11, the dataset has 68 reported rat sightings on this day.

On day 2026-03-10, the model predicts 40 rat sightings reported to 311.

On day 2026-03-11, the model predicts 39 rat sightings reported to 311.

On day 2026-03-12, the model predicts 41 rat sightings reported to 311.

On day 2026-03-13, the model predicts 37 rat sightings reported to 311.

On day 2026-03-14, the model predicts 20 rat sightings reported to 311.

On day 2026-03-15, the model predicts 22 rat sightings reported to 311.

On day 2026-03-16, the model predicts 45 rat sightings reported to 311.

On day 2026-03-17, the model predicts 41 rat sightings reported to 311.

On day 2026-03-18, the model predicts 42 rat sightings reported to 311.

On day 2026-03-19, the model predicts 47 rat sightings reported to 311.

On day 2026-03-20, the model predicts 36 rat sightings reported to 311.

On day 2026-03-21, the model predicts 24 rat sightings reported to 311.

On day 2026-03-22, the model predicts 22 rat sightings reported to 311.

## Plots 

In [131]:
model.plot(forecast)

ERROR - (NP.plotly.plot) - plotly-resampler is not installed. Please install it to use the resampler.
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/plot_forecast_plotly.py:100: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result




In [132]:
future = model.make_future_dataframe(rs)
forecast = model.predict(rs)
model.plot(forecast)


INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/data/split.py:273: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_ut

Predicting: 18it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
ERROR - (NP.plotly.plot) - plotly-resampler is not installed. Please install it to use the resampler.
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/plot_forecast_plotly.py:100: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result




## Hyperparamter Tuning

Uncomment the following codeblock if one wishes to tune the hyperparameters above. Only do so if absolutely necessary due to computational complexity. The codeblock will then print out the best parameters it found. 

**final_model_neural.db** Optuna is able to save its previous trials and study to a file and reload its studies one future runs. This information is held inside of "final_model_neural.db". Optuna was designed with flexibility in mind e.g. adding new parameters to teach trial, changing the suggested values for each trial, and changing the sizes of the time series split will not break anything.

In [136]:
# Open rat sightings data
rat_sighting = pd.read_csv("../../scr/data/rat_sightings_data/Rat_Sightings_NYC.csv")



# Clean the data
rat_sighting.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rat_sighting.columns] #apply to column headers
rat_sighting['location_type'] = rat_sighting['location_type'].str.strip().str.replace(' ', '_').str.lower()  #apply to location_type column
cols_to_drop = [c for c in rat_sighting.columns if (rat_sighting[c].nunique(dropna=False) == 1)]
rat_sighting = rat_sighting.drop(columns=cols_to_drop)
rat_sighting['created_date'] = pd.to_datetime(rat_sighting['created_date']) 
rat_sighting = rat_sighting.drop(columns='park_borough')
rat_sighting = rat_sighting.drop(columns=['location'])
rs = rat_sighting.copy()
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

# Download the weather data

lat, lon = 40.7831, -73.9712
last_date = rs['ds'].max()
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)


# Add the weather data to rs

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])

rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")



# Time Series Split. Set to n_splits = 2 to reduce run time. Make higher to avoid overfitting.
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=2, test_size=14)


def objective(trial):
    regressor_lags = {
        'apparent_temperature_max': trial.suggest_int('lag_temp_max', 1, 365),
        'apparent_temperature_min': trial.suggest_int('lag_temp_min', 1, 365),
        'snowfall_sum': trial.suggest_int('lag_snowfall', 1, 365),
    }
    n_lags = trial.suggest_int('n_lags', 14, 365)
    epochs = trial.suggest_int('epochs', 50, 1000)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.5, log=True)
    batch_size = trial.suggest_int('batch_size', 20, 256)
    ar_reg = trial.suggest_float('ar_reg', 0.5, 10)
    fold_rmses = []
    for i, (train_idx, test_idx) in enumerate(tscv.split(rs)):

        train = rs.iloc[train_idx].copy()
        test = rs.iloc[test_idx].copy()
        
        existing_regressors = [col for col in regressed_features if col in train.columns]
        train = train.dropna(subset=["y"] + existing_regressors)
        test = test.dropna(subset=existing_regressors)
        
        # Skip fold if too few rows
        if len(train) < 20 or len(test) < 1:
            continue
        
        model = NeuralProphet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            n_lags=n_lags,
            epochs=epochs,
            ar_reg = ar_reg,
            accelerator="auto",   # uses GPU if available
            learning_rate=learning_rate,
            batch_size=batch_size
        )
        model.add_country_holidays(country_name="US")
        for col in existing_regressors:
            model.add_lagged_regressor(col, n_lags=regressor_lags[col])
        
        model.fit(train, freq="D", progress="off")
        future = pd.concat([
            train[['ds','y'] + existing_regressors],
            test[['ds','y']].merge(wd[['ds'] + existing_regressors], on="ds", how="left")
        ])
        future = future.dropna(subset=existing_regressors)
        forecast = model.predict(future)
        
        y_pred = forecast["yhat1"].iloc[-len(test):].values
        y_true = test["y"].values
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        fold_rmses.append(rmse)

        # intermediate_score = np.mean(fold_rmses)
        # trial.report(intermediate_score, i)
        # if trial.should_prune():
        #     raise optuna.TrialPruned()
        
    return np.mean(fold_rmses) if fold_rmses else float("inf")

study = optuna.create_study(
    direction="minimize",
    study_name="final_model_neural",
    storage="sqlite:///final_model_neural.db",
    load_if_exists=True
)
study.optimize(objective, n_trials=1)  # adjust n_trials as needed



best_params = study.best_params

WARNING - (py.warnings._showwarnmsg) - /var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_48662/3724533446.py:11: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


[I 2026-03-11 11:52:34,574] Using an existing study with name 'final_model_neural' instead of creating a new one.
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.955% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.utils.con

Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.955% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.955% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 28it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.955% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.utils.configure_trainer) - Using accelerator mps with 1 device(s).


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 28it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
[I 2026-03-11 12:00:53,410] Trial 24 finished with value: 14.01331071479684 and parameters: {'lag_temp_max': 95, 'lag_temp_min': 84, 'lag_snowfall': 85, 'n_lags': 277, 'epochs': 376, 'learning_rate': 0.014957033705943323, 'batch_size': 72, 'ar_reg': 1.625347018404393}. Best is trial 0 with value: 9.91822616443868.
